# Ad Impressions Forecasting

**Project 8 of 10** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **hourly ad impression volumes** using the Kaggle *Avazu CTR Prediction* dataset. We aggregate click-log records to hourly impression counts (every record = 1 impression) and apply time-series forecasting.

| Attribute | Value |
|---|---|
| **Target** | `impressions` (hourly count) |
| **Frequency** | Hourly (`h`) |
| **Seasonal period** | 24 (daily cycle) |
| **Primary TS library** | MLForecast |
| **Kaggle competition** | `avazu-ctr-prediction` |

### Why MLForecast?
Ad impression data has strong hour-of-day seasonality. MLForecast automates lag/rolling feature creation and recursive multi-step forecasting.

## Learning Objectives

1. Parse a large click-log dataset with YYMMDDHH encoded timestamps
2. Aggregate raw records to **hourly impression counts**
3. Explore **diurnal** and **day-of-week** patterns in ad traffic
4. Build naive and seasonal naive (24h) baselines
5. Benchmark via LazyPredict on lag features
6. Run FLAML AutoML
7. Train MLForecast models (LightGBM, XGBoost, Ridge)
8. Evaluate with MAE, RMSE, MAPE

## Problem Statement

Given 10 days of click-level ad impression logs (~40M records), **forecast hourly ad impression volume for the next 24 hours**.

Ad platforms need impression forecasts to price inventory, allocate server capacity, and detect anomalous traffic.

## Why This Project Matters

- **Revenue optimization**: Ad pricing depends on expected impression volume
- **Capacity planning**: Ad servers must handle peak hourly traffic
- **Fraud detection**: Deviations from forecast signal bot traffic or click fraud
- **Campaign planning**: Advertisers budget campaigns based on impression forecasts

## Dataset Overview

| File | Description |
|------|-------------|
| `train.gz` | ~40M click/impression records over 10 days |

### Key columns
- **hour** — YYMMDDHH encoded (e.g., 14102100 = 2014-10-21 00:00)
- **click** — 1 if clicked, 0 if not (every record = 1 impression)

### Derived target
- **impressions** = `count(*)` per hour — total impression volume each hour

## Dataset Source & License Notes

- **Kaggle**: [Avazu CTR Prediction](https://www.kaggle.com/competitions/avazu-ctr-prediction)
- **License**: Competition rules — educational use only

## Environment Setup

In [ ]:
import subprocess, sys
def _install(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly","scikit-learn","lazypredict","flaml","mlforecast","lightgbm","xgboost","statsmodels","scipy","window-ops"]:
    try: __import__(pkg.replace("-","_"))
    except ImportError: _install(pkg)
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px, plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from mlforecast import MLForecast
import lightgbm as lgb, xgboost as xgb
from window_ops.rolling import rolling_mean, rolling_std
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Ad Impressions Forecasting"
KAGGLE_SLUG  = "avazu-ctr-prediction"
TARGET  = "impressions"
FREQ    = "h"
SEASON_LENGTH    = 24
FORECAST_HORIZON = 24
TEST_SIZE  = FORECAST_HORIZON
VAL_SIZE   = FORECAST_HORIZON
RANDOM_STATE = 42
FLAML_BUDGET = 60
MAX_ROWS = 5_000_000
print(f"Project: {PROJECT_NAME} | Target: {TARGET} | Horizon: {FORECAST_HORIZON}h")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True
if not kaggle_ok:
    raise RuntimeError(
        "No Kaggle credentials found!\n"
        "Set KAGGLE_API_TOKEN env var, or place kaggle.json in ~/.kaggle/\n"
        "See: https://www.kaggle.com/settings -> Create New Token"
    )
print("Ready to download.")

## Dataset Download & Loading

Load a 5M-row sample and aggregate to hourly impression counts.

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.makedirs("data", exist_ok=True)
    ret = os.system(f"kaggle competitions download -c {KAGGLE_SLUG} -p data/ --unzip")
    if ret != 0: raise RuntimeError("Download failed.")
    data_path = pathlib.Path("data")
for f in sorted(data_path.rglob("*")):
    if f.is_file(): print(f"  {f.name:40s}  {f.stat().st_size/1e6:7.2f} MB")

In [ ]:
train_files = [f for f in data_path.rglob("*") if f.is_file() and "train" in f.name.lower()]
assert train_files, "No training file found!"
raw = pd.read_csv(train_files[0], nrows=MAX_ROWS)
print(f"Loaded: {raw.shape[0]:,} rows x {raw.shape[1]} cols")
raw.head(3)

## Data Validation Checks

In [ ]:
print("=" * 60 + "\nDATA VALIDATION\n" + "=" * 60)
assert "hour" in raw.columns, "Missing hour column!"
assert "click" in raw.columns, "Missing click column!"
print(f"  Rows: {raw.shape[0]:,}  Cols: {raw.shape[1]}")
print(f"  Hour range: {raw['hour'].min()} to {raw['hour'].max()}")
print(f"  Unique hours: {raw['hour'].nunique()}")
print(f"  Click rate: {raw['click'].mean()*100:.2f}%")
print(f"  NaN in hour: {raw['hour'].isna().sum()}")
print("Validation complete.")

## Exploratory Data Analysis

Decode timestamps, aggregate to hourly impressions, explore patterns.

In [ ]:
raw["datetime"] = pd.to_datetime(raw["hour"].astype(str), format="%y%m%d%H")
hourly = (
    raw.groupby("datetime")
    .agg(impressions=("click", "count"), clicks=("click", "sum"))
    .reset_index().sort_values("datetime").reset_index(drop=True)
)
hourly["ctr"] = hourly["clicks"] / hourly["impressions"]
fr = pd.date_range(hourly["datetime"].min(), hourly["datetime"].max(), freq="h")
hourly = hourly.set_index("datetime").reindex(fr).rename_axis("datetime").reset_index()
hourly["impressions"] = hourly["impressions"].interpolate()
print(f"Hourly series: {len(hourly)} hours")
print(f"Range: {hourly['datetime'].min()} to {hourly['datetime'].max()}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=hourly["datetime"], y=hourly["impressions"], name="Hourly", line=dict(width=1)))
fig.add_trace(go.Scatter(x=hourly["datetime"], y=hourly["impressions"].rolling(24).mean(), name="24h MA", line=dict(color="red", width=2)))
fig.update_layout(title="Avazu — Hourly Ad Impressions", template="plotly_white")
fig.show()

In [ ]:
hourly["hour_of_day"] = hourly["datetime"].dt.hour
fig = px.box(hourly, x="hour_of_day", y="impressions", title="Impressions by Hour of Day")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
if len(hourly) > 2 * SEASON_LENGTH:
    ts = hourly.set_index("datetime")["impressions"].asfreq("h").interpolate()
    decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    decomp.observed.plot(ax=axes[0], title="Observed")
    decomp.trend.plot(ax=axes[1], title="Trend")
    decomp.seasonal.plot(ax=axes[2], title="Daily Seasonal (24h)")
    decomp.resid.plot(ax=axes[3], title="Residual")
    plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
print("Impression Volume Statistics:")
print(hourly["impressions"].describe().to_string())
print(f"\nSkewness: {hourly['impressions'].skew():.3f}")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(hourly["impressions"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Hourly Impressions")
axes[1].boxplot(hourly["impressions"].dropna())
axes[1].set_title("Box Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split Strategy

Strict temporal split: last 24h = test, preceding 24h = validation, rest = train.

In [ ]:
ts_df = hourly[["datetime","impressions"]].rename(columns={"datetime":"ds","impressions":"y"}).copy()
n = len(ts_df)
ts_train = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_val = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_test = ts_df.iloc[n-TEST_SIZE:].copy()
print(f"Train: {len(ts_train)}  Val: {len(ts_val)}  Test: {len(ts_test)}")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("No temporal overlap.")
ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train"))
fig.add_trace(go.Scatter(x=ts_val["ds"], y=ts_val["y"], name="Val", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Test", line=dict(color="red")))
fig.update_layout(title="Temporal Split", template="plotly_white")
fig.show()

## Preprocessing Strategy

No scaling for tree models. Missing hours interpolated. Lag features prevent leakage via `.shift()`.

## Feature Engineering

In [ ]:
def make_lag_features(df):
    out = df.copy()
    for lag in [1, 24, 48, 168]: out[f"lag_{lag}"] = out["y"].shift(lag)
    for w in [24, 48]:
        out[f"rmean_{w}"] = out["y"].shift(1).rolling(w).mean()
        out[f"rstd_{w}"]  = out["y"].shift(1).rolling(w).std()
    out["hour"] = out["ds"].dt.hour
    out["dow"]  = out["ds"].dt.dayofweek
    out["day"]  = out["ds"].dt.day
    return out
feat = make_lag_features(ts_df).dropna().reset_index(drop=True)
fcols = [c for c in feat.columns if c not in ["ds","y"]]
tc = ts_train["ds"].max(); vc = ts_val["ds"].max()
ft = feat[feat["ds"]<=tc]; fv = feat[(feat["ds"]>tc)&(feat["ds"]<=vc)]; fte = feat[feat["ds"]>vc]
X_tr,y_tr = ft[fcols],ft["y"]; X_v,y_v = fv[fcols],fv["y"]; X_te,y_te = fte[fcols],fte["y"]
X_tv = pd.concat([X_tr,X_v]); y_tv = pd.concat([y_tr,y_v])
print(f"X_train:{X_tr.shape}  X_val:{X_v.shape}  X_test:{X_te.shape}")

## Baseline Model

In [ ]:
def calc_metrics(actual, predicted, name="Model"):
    actual, predicted = np.array(actual,float), np.array(predicted,float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask]-predicted[mask])/actual[mask]))*100 if mask.any() else np.nan
    return {"Model":name,"MAE":round(mae,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}
results = []; actual_test = ts_test["y"].values
results.append(calc_metrics(actual_test, np.full(TEST_SIZE, ts_trainval["y"].iloc[-1]), "Naive"))
sn = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
results.append(calc_metrics(actual_test, np.tile(sn,2)[:TEST_SIZE], "Seasonal Naive (24h)"))
results.append(calc_metrics(actual_test, np.full(TEST_SIZE, ts_trainval["y"].iloc[-24:].mean()), "MA(24)"))
print("Baselines:"); print(pd.DataFrame(results).to_string(index=False))

## LazyPredict Benchmark

LazyPredict on lag features — tabular regression benchmark, not native forecasting.

In [ ]:
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=RANDOM_STATE)
    lm, _ = lazy.fit(X_tr, X_v, y_tr, y_v)
    print("LazyPredict top 15:"); print(lm.head(15).to_string())
    for i,(nm,row) in enumerate(lm.head(3).iterrows()):
        results.append({"Model":f"LP: {nm}","MAE":round(row.get("MAE",0),2),"RMSE":round(row.get("RMSE",0),2),"MAPE":np.nan})
except Exception as e: print(f"LazyPredict failed: {e}")

## FLAML AutoML Run

In [ ]:
try:
    fl = AutoML()
    fl.fit(X_train=X_tv, y_train=y_tv, task="regression", time_budget=FLAML_BUDGET, metric="rmse", verbose=0, seed=RANDOM_STATE)
    fp = fl.predict(X_te)
    results.append(calc_metrics(actual_test[:len(fp)], fp, f"FLAML ({fl.best_estimator})"))
    print(f"FLAML best: {fl.best_estimator}")
except Exception as e: print(f"FLAML failed: {e}")

## Top 3 Model Selection

In [ ]:
results_df_mid = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("Current Ranking:"); print(results_df_mid.to_string(index=False))

## MLForecast — Dedicated Time-Series Models

MLForecast handles recursive forecasting with lag-24 as the key feature for hourly ad impressions.

In [ ]:
mlf_data = ts_trainval.copy()
mlf_data["unique_id"] = "total_impressions"
mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=RANDOM_STATE, verbosity=-1),
        "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE, verbosity=0),
        "Ridge": Ridge(alpha=1.0),
    },
    freq="h", lags=[1, 24, 48, 168],
    lag_transforms={1: [(rolling_mean, 24), (rolling_std, 24)], 24: [(rolling_mean, 7)]},
    date_features=["hour", "dayofweek", "day"],
)
mlf.fit(mlf_data)
mlf_preds = mlf.predict(h=FORECAST_HORIZON)
print(f"MLForecast: {mlf_preds.shape}"); mlf_preds.head()

In [ ]:
for mn in ["LightGBM", "XGBoost", "Ridge"]:
    if mn in mlf_preds.columns:
        pv = mlf_preds[mn].values[:TEST_SIZE]
        r = calc_metrics(actual_test, pv, f"MLF: {mn}")
        results.append(r)
        print(f"{mn}: MAE={r['MAE']}, RMSE={r['RMSE']}, MAPE={r['MAPE']}%")

## Final Training & Evaluation of Top 3

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("ALL MODELS:"); print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\nTOP 3:"); print(top3.to_string(index=False))
fig = px.bar(results_df, x="Model", y="RMSE", title="Model Comparison", color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

## Forecast vs Actual — Test Period

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=actual_test, name="Actual", line=dict(color="black", width=3)))
for mn in ["LightGBM","XGBoost","Ridge"]:
    if mn in mlf_preds.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=mlf_preds[mn].values[:TEST_SIZE], name=f"MLF: {mn}", line=dict(dash="dash")))
fig.update_layout(title="Forecast vs Actual (24h)", template="plotly_white")
fig.show()

## Error Analysis

In [ ]:
best_col = "LightGBM" if "LightGBM" in mlf_preds.columns else mlf_preds.columns[-1]
best_pred = mlf_preds[best_col].values[:TEST_SIZE]
errors = actual_test - best_pred
pct_err = np.where(actual_test != 0, errors/actual_test*100, 0)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(errors, bins=15, edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Error Distribution")
axes[1].plot(range(1,TEST_SIZE+1), np.abs(pct_err), marker="o"); axes[1].set_title("|% Error| by Hour")
axes[2].scatter(actual_test, best_pred, alpha=0.7)
mn_,mx_ = min(actual_test.min(),best_pred.min()), max(actual_test.max(),best_pred.max())
axes[2].plot([mn_,mx_],[mn_,mx_],"r--"); axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()
print(f"Bias: {errors.mean():,.0f}  MAE: {np.abs(errors).mean():,.0f}")

## Interpretation & Business Insight

1. **Diurnal cycle dominates**: Ad impressions peak during waking hours, trough overnight
2. **Lag-24 is the strongest feature**: Same hour yesterday is the best predictor
3. **Day-of-week matters**: Weekend vs weekday traffic patterns differ
4. **Revenue link**: Impression forecasts × CPM = revenue projections
5. **Short series challenge**: Only ~10 days limits model complexity

## Limitations

1. Short history (~10 days of training data)
2. Sampled 5M of ~40M records — full data may differ
3. No ad context features (site, app, device)
4. Aggregated view — per-site forecasts would be more actionable
5. Non-stationary traffic evolves over time

## How to Improve This Project

1. Load full dataset for complete hourly aggregation
2. Per-site panel forecasting via MLForecast
3. Add campaign schedule features as external regressors
4. Combine with CTR model for click volume forecasts
5. Rolling-origin cross-validation

## Production Considerations

1. Hourly retraining with fresh data
2. Low-latency serving for real-time bidding
3. Anomaly alerting when actuals deviate >30%
4. Revenue projection integration
5. Fallback to seasonal naive if model fails

## Common Mistakes to Avoid

1. Including future data in features (always `.shift()`)
2. Random train/test splits instead of temporal
3. Confusing CTR (rate) with impression volume (count)
4. Over-aggregating to daily totals (losing hourly peaks)
5. MAPE with zero-impression hours

## Mini Challenge / Exercises

1. Per-site forecasting with `unique_id = site_id`
2. Click forecasting instead of impressions
3. Log transform (`np.log1p`) and compare MAPE
4. Extend horizon to 48 hours
5. Ensemble LightGBM + XGBoost

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Avazu CTR** dataset and decoded YYMMDDHH timestamps
- Aggregated to **hourly impression counts**
- Explored diurnal and day-of-week patterns
- Built baselines (naive, seasonal naive 24h, MA)
- Benchmarked via LazyPredict and FLAML
- Trained MLForecast models (LightGBM, XGBoost, Ridge)
- Ranked all models and analyzed errors

### Key Takeaways
1. **24h diurnal cycle is the dominant pattern** in ad impressions
2. **MLForecast with lag-24** effectively captures hourly patterns
3. **Short series** (~10 days) limits sophistication — keep models simple
4. **Per-site forecasting** is the natural production extension

---
*Time-series forecasting portfolio — educational purposes only.*